# 🗄️ Notebook 02: SQL Database Schema Setup & Analytical Queries
**Objective:** Load cleaned datasets into an SQLite database, create structured relational tables, and run advanced analytical SQL queries (window functions, YoY growth, milestone aggregations) to validate metrics before Power BI ingestion.

In [2]:
import sqlite3
import pandas as pd

# Connect to SQLite database (creates 'virat_kohli_analytics.db' if it doesn't exist)
conn = sqlite3.connect('virat_kohli_analytics.db')
cursor = conn.cursor()

# Load cleaned CSV datasets created in Notebook 01
df_yearly_cleaned = pd.read_csv('cleaned_virat_international_yearwise_2026.csv')
df_milestones_cleaned = pd.read_csv('cleaned_virat_milestones_2026.csv')

# Push DataFrames to SQL database tables
df_yearly_cleaned.to_sql('fact_yearly_performance', conn, if_exists='replace', index=False)
df_milestones_cleaned.to_sql('fact_career_milestones', conn, if_exists='replace', index=False)

print("✅ Data successfully loaded into SQLite database tables!")

✅ Data successfully loaded into SQLite database tables!


## Step 1: Database DDL Schemas 
Below are standard SQL DDL scripts if you want to deploy these tables to **PostgreSQL / MySQL / MS SQL Server**.

In [7]:
# Wrap SQL commands inside sqlite3 executescript
ddl_script = """
CREATE TABLE IF NOT EXISTS fact_yearly_performance (
    performance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    Player TEXT,
    Format TEXT,
    Year INTEGER,
    Matches INTEGER,
    Innings INTEGER,
    Runs INTEGER,
    Average REAL,
    Strike_Rate REAL,
    Fifties INTEGER,
    Hundreds INTEGER,
    Highest_Score TEXT,
    Is_Not_Out BOOLEAN,
    Highest_Score_Clean INTEGER,
    Conversion_Rate_Pct REAL,
    Runs_Per_Innings REAL,
    Career_Era TEXT
);
"""

cursor.executescript(ddl_script)
conn.commit()
print("Table created via SQL script!")

Table created via SQL script!


## Step 2: SQL Analytical Queries Execution
Helper function to execute SQL queries using `pandas` and display formatted query output.

In [10]:
def run_sql(query):
    """Executes SQL query and returns result as a Pandas DataFrame."""
    return pd.read_sql_query(query, conn)

print("SQL execution helper function defined.")

SQL execution helper function defined.


### Query 1: Format-Wise Career Totals & Averages
Summarizes overall career numbers (Matches, Runs, Weighted Batting Average, Centuries, and Highest Score) grouped by format.

In [13]:
query_1 = """
SELECT 
    Format,
    SUM(Matches) AS Total_Matches,
    SUM(Innings) AS Total_Innings,
    SUM(Runs) AS Total_Runs,
    ROUND(CAST(SUM(Runs) AS FLOAT) / NULLIF(SUM(Innings), 0), 2) AS Overall_Batting_Average,
    SUM(Hundreds) AS Total_Centuries,
    SUM(Fifties) AS Total_Fifties,
    MAX(Highest_Score_Clean) AS Highest_Score
FROM fact_yearly_performance
GROUP BY Format
ORDER BY Total_Runs DESC;
"""

display(run_sql(query_1))

,Format,Total_Matches,Total_Innings,Total_Runs,Overall_Batting_Average,Total_Centuries,Total_Fifties,Highest_Score
0,ODI,311,305,14797,48.51,54,77,183
1,Test,123,201,9230,45.92,30,31,254
2,T20I,125,131,4188,31.97,1,38,122


### Query 2: Year-over-Year (YoY) Run Growth Analysis
Uses the `LAG()` window function to track yearly run trajectory and calculate percentage growth year-over-year.

In [16]:
query_2 = """
WITH YearlyTotals AS (
    SELECT 
        Year,
        SUM(Runs) AS Total_Runs_Year,
        SUM(Hundreds) AS Centuries_Year
    FROM fact_yearly_performance
    GROUP BY Year
)
SELECT 
    Year,
    Total_Runs_Year,
    Centuries_Year,
    LAG(Total_Runs_Year, 1, 0) OVER (ORDER BY Year) AS Previous_Year_Runs,
    ROUND(
        (Total_Runs_Year - LAG(Total_Runs_Year, 1) OVER (ORDER BY Year)) * 100.0 / 
        NULLIF(LAG(Total_Runs_Year, 1) OVER (ORDER BY Year), 0), 2
    ) AS YoY_Run_Growth_Pct
FROM YearlyTotals
ORDER BY Year;
"""

display(run_sql(query_2))

,Year,Total_Runs_Year,Centuries_Year,Previous_Year_Runs,YoY_Run_Growth_Pct
0,2008,159,0,0,NaN
1,2009,325,1,159,104.40
2,2010,1021,3,325,214.15
3,2011,1771,4,1021,73.46
4,2012,1900,8,1771,7.28
5,2013,2139,6,1900,12.58
6,2014,2286,8,2139,6.87
7,2015,1336,4,2286,-41.56
8,2016,2595,7,1336,94.24
9,2017,2818,11,2595,8.59


### Query 3: Top Opponents by Milestone Count
Aggregates career achievements against major opponent teams across ODI, Test, and T20I formats.

In [19]:
query_3 = """
SELECT 
    Opponent_or_Event,
    COUNT(*) AS Total_Milestones,
    SUM(CASE WHEN Format = 'ODI' THEN 1 ELSE 0 END) AS ODI_Milestones,
    SUM(CASE WHEN Format = 'Test' THEN 1 ELSE 0 END) AS Test_Milestones,
    SUM(CASE WHEN Format = 'T20I' THEN 1 ELSE 0 END) AS T20I_Milestones
FROM fact_career_milestones
GROUP BY Opponent_or_Event
ORDER BY Total_Milestones DESC
LIMIT 10;
"""

display(run_sql(query_3))

,Opponent_or_Event,Total_Milestones,ODI_Milestones,Test_Milestones,T20I_Milestones
0,vs Sri Lanka,2,2,0,0
1,vs South Africa,1,0,0,1
2,vs New Zealand,1,1,0,0
3,vs Australia,1,0,1,0
4,vs Afghanistan,1,0,0,1
5,973 runs in season,1,0,0,0
6,"842 matches, 37551 runs, 214 fifties, 94 hundreds",1,0,0,0
7,"559 matches, 28215 runs, 146 fifties, 85 hundreds",1,0,0,0
8,"283 matches, 9336 runs, 68 fifties, 9 hundreds...",1,0,0,0


## Step 3: Close SQL Database Connection
Complete the database session cleanly.

In [22]:
conn.close()
print("🔒 Database connection closed successfully. Notebook 02 complete!")

🔒 Database connection closed successfully. Notebook 02 complete!
